In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [2]:
pth = os.path.join('..', '..', '..')
pth_data = os.path.join(pth, 'data')
pth_in = os.path.join(pth_data, 'cancer', 'CCCA')
pth_ref = os.path.join(pth_data, 'ref', 'AnimalTFDBv4')
# pth_fig = os.path.join(pth, 'figures', 'development', 'regulation')
# pth_out = os.path.join(pth_data, 'results')

In [3]:
adata = sc.read_h5ad(os.path.join(pth_in, 'spectrum.h5ad'))
adata

AnnData object with n_obs × n_vars = 149645 × 513
    obs: 'sample', 'cell_type', 'complexity', 'umap1', 'umap2', 'g1s_score', 'g2m_score', 'cell_cycle_phase', 'mp_top_score', 'mp_top', 'mp_assignment', 'ebv', 'celltype', 'Title', 'Category', 'Disease', 'Technology', 'Samples', 'Cells', 'Name', 'patient', 'cell_subtype', 'source', 'disease', 'p16_status', 'site', 't_stage', 'n_stage', 'gender', 'race', 'age', 'tobacco_use', 'alcohol_use', 'hpv', 'hpv_score', 'treatment', 'procedure', 'sorting', 'cluster', 'malignant', 'sample_id', 'cell_subtype_clusters', 'tnm_stage', 'viral', 'location', 'cancer_type', 'treatment_exposure', 'treatment_response', 'cell_lineage', 'type', 'clusters_by_authors', 'cell_QCpass', 'is_tumor', 'cell_compartment', 'cluster_assignment', 'metastasis', 'sex', 'prior_enzalutamide', 'prior_abiraterone', 'prior_taxane', 'prior_platinum', 'prior_sipuleucel_T', 'purity', 'has_bulk_RNA', 'cancer_cell_state', 'PSA', 'source_region', 'gleason_score', 'ERG_status', 'cores'

In [4]:
# transcription factors
fn = os.path.join(pth_ref, 'Homo_sapiens_TF')
tf = pd.read_csv(fn, sep = '\t').Symbol
tf = tf.loc[tf.isin(adata.var_names)].values

In [5]:
n = adata.obs.source.value_counts()
src = n.index[n > 100]
msk = adata.obs.source.isin(src)
adata = adata[msk].copy()
adata

AnnData object with n_obs × n_vars = 148719 × 513
    obs: 'sample', 'cell_type', 'complexity', 'umap1', 'umap2', 'g1s_score', 'g2m_score', 'cell_cycle_phase', 'mp_top_score', 'mp_top', 'mp_assignment', 'ebv', 'celltype', 'Title', 'Category', 'Disease', 'Technology', 'Samples', 'Cells', 'Name', 'patient', 'cell_subtype', 'source', 'disease', 'p16_status', 'site', 't_stage', 'n_stage', 'gender', 'race', 'age', 'tobacco_use', 'alcohol_use', 'hpv', 'hpv_score', 'treatment', 'procedure', 'sorting', 'cluster', 'malignant', 'sample_id', 'cell_subtype_clusters', 'tnm_stage', 'viral', 'location', 'cancer_type', 'treatment_exposure', 'treatment_response', 'cell_lineage', 'type', 'clusters_by_authors', 'cell_QCpass', 'is_tumor', 'cell_compartment', 'cluster_assignment', 'metastasis', 'sex', 'prior_enzalutamide', 'prior_abiraterone', 'prior_taxane', 'prior_platinum', 'prior_sipuleucel_T', 'purity', 'has_bulk_RNA', 'cancer_cell_state', 'PSA', 'source_region', 'gleason_score', 'ERG_status', 'cores'

In [6]:
adata_src = sc.get.aggregate(adata, 'source', 'mean')

In [ ]:
# X: TF expression, source
src = adata_src.obs_names.copy()
X = adata_src[:, tf].layers['mean']
X = StandardScaler().fit_transform(X)
src = pd.get_dummies(src).values.astype(float)
X = np.concatenate((X, src), axis = 1)

# y: late signature std ()
y = (adata.obs.groupby('source', observed = True)
     .late.std().values.reshape(-1, 1))
y = StandardScaler().fit_transform(y).flatten()

# linear regression (y ~ X)
reg = LinearRegression(fit_intercept = False).fit(X, y)

# TF coefficients (beta)
beta = pd.Series(reg.coef_[:-src.shape[1]], index = tf)
beta = beta.sort_values(ascending = False); beta.name = 'coef'
# beta.to_csv(os.path.join(pth_out, 'reg_coef.csv')); 
beta

PRRX1    0.211577
FOXP1    0.174997
HOXB8    0.172338
TBXT     0.155405
SNAI2    0.148269
           ...   
SOX4    -0.103827
SP5     -0.110205
FOXB1   -0.113421
FOXC2   -0.114428
ID1     -0.119125
Name: coef, Length: 98, dtype: float64

In [ ]:
# cmap_dict = {
#     'Pou5f1' : 'Purples',
#     'Prrx1'  : 'Greens'}

In [ ]:
# geo = 'GSE162534'
# vlim = {'Pou5f1' : [3.85, 'p92'], 'Prrx1' : [.8, 'p50']}
# adata_plt = adata[adata.obs.source == geo].copy()

# fig, ax = plt.subplots(1, 1, figsize = (3.5, 1.5))
# sc.pl.embedding(adata, 'X_spectrum', size = 80, alpha = .5,
#                 frameon = False, show = False, ax = ax)
# for g, (vmin, vmax) in vlim.items():
#     sc.pl.embedding(adata_plt[adata_plt[:, g].X.toarray() > vmin],
#                     'X_spectrum', size = 80, color = g,
#                     cmap = cmap_dict[g], vmin = vmin,
#                     vmax = vmax, colorbar_loc = None,
#                     frameon = False, show = False, ax = ax)
# ax.set_title('')
# fn = os.path.join(pth_fig, f'{geo}.pdf')
# plt.savefig(fn, bbox_inches = 'tight')

In [ ]:
# geo = 'GSE229103'
# vlim = {'Prrx1' : [.5, 'p70'], 'Pou5f1' : [.7, 'p75']}
# adata_plt = adata[adata.obs.source == geo].copy()

# fig, ax = plt.subplots(1, 1, figsize = (3.5, 1.5))
# sc.pl.embedding(adata, 'X_spectrum', size = 80, alpha = .5,
#                 frameon = False, show = False, ax = ax)
# for g, (vmin, vmax) in vlim.items():
#     sc.pl.embedding(adata_plt[adata_plt[:, g].X.toarray() > vmin],
#                     'X_spectrum', size = 80, color = g,
#                     cmap = cmap_dict[g], vmin = vmin,
#                     vmax = vmax, colorbar_loc = None,
#                     frameon = False, show = False, ax = ax)
# ax.set_title('')
# fn = os.path.join(pth_fig, f'{geo}.pdf')
# plt.savefig(fn, bbox_inches = 'tight')

In [ ]:
# geo = 'rRNAModifications'
# vlim = {'Pou5f1' : [0, 'p95'], 'Prrx1' : [1.25, 'p95']}
# adata_plt = adata[adata.obs.source == geo].copy()

# fig, ax = plt.subplots(1, 1, figsize = (3.5, 1.5))
# sc.pl.embedding(adata, 'X_spectrum', size = 80, alpha = .5,
#                 frameon = False, show = False, ax = ax)
# for g, (vmin, vmax) in vlim.items():
#     sc.pl.embedding(adata_plt[adata_plt[:, g].X.toarray() > vmin],
#                     'X_spectrum', size = 80, color = g,
#                     cmap = cmap_dict[g], vmin = vmin,
#                     vmax = vmax, colorbar_loc = None,
#                     frameon = False, show = False, ax = ax)
# ax.set_title('')
# fn = os.path.join(pth_fig, f'{geo}.pdf')
# plt.savefig(fn, bbox_inches = 'tight')